# 1장 — 큰 그림: 챗봇에서 에이전트로

웹 챕터(1장)와 **같은 예제**인 "다음 주 화요일 회의실 예약" 요청을 챗봇과 에이전트로 처리해 보며
**에이전트 = LLM + 도구 + 루프**를 파이썬 최소 코드로 해부합니다. 난수 없는 완전 결정적 예제예요.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch1')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 챗봇 모드: 한 턴에 답하는 함수

챗봇은 **입력 → 출력 한 번**으로 끝나는 함수입니다. 외부 세계(캘린더)를 읽을 손이 없어서,
훈련 때 본 글로 **그럴듯한 문장**을 지어낼 수밖에 없어요.

In [ ]:
def chatbot(request):
    """한 턴 함수 — 캘린더에 접근할 수단이 없다."""
    return '네! 다음 주 화요일 오후에 회의실을 예약해 드렸습니다 😊'

print('요청: 다음 주 화요일에 팀 회의실 좀 예약해 줘')
print('챗봇:', chatbot('회의실 예약'))
print('⚠️ 실제로 예약된 것은 없다. 캘린더를 본 적조차 없음!')

## 2. 에이전트 모드: 관찰→생각→행동 루프 + 도구

도구(tool)는 **외부 손**입니다. 여기서는 파이썬 dict 로 만든 가짜 캘린더·회의실 시스템.
에이전트는 스텝마다 **생각(왜) → 행동(도구 호출) → 관찰(결과)** 을 반복하고,
그 기록 전체가 **궤적(trajectory)** 이 됩니다.

In [ ]:
# ── 도구들: 결정적 가짜 사내 시스템 ──────────────────────────────
CALENDAR = {'다음주_화': {'15시': ['팀원A', '팀원B', '팀원C', '팀원D']}}  # 전원 가능 시간
ROOMS = {('다음주_화', '15시'): 'B동 3호'}                              # 빈 회의실
BOOKINGS = {}

def tool_calendar(day):            return f"{day} 15시, 팀원 4명 전원 비어 있음"
def tool_find_room(day, time):     return f"{ROOMS[(day, time)]}가 {time}~16시 비어 있음"
def tool_book(room, day, time):    BOOKINGS['R-1027'] = (room, day, time); return '예약 확정: 번호 R-1027'
def tool_invite(members):          return f'{members}명 모두에게 초대장 발송됨'
def tool_confirm(booking_id):      return f"상태: 확정 ({BOOKINGS[booking_id][0]})"

# ── 에이전트 루프: 생각 → 행동(도구) → 관찰 ─────────────────────
trajectory = []

def step(think, action_name, result):
    trajectory.append({'생각': think, '행동': action_name, '관찰': result})
    print(f"스텝 {len(trajectory)} | 💭 {think}")
    print(f"        | 🛠️ {action_name}  →  👀 {result}")

step('언제, 몇 명인지부터 확인', '캘린더 조회(다음주_화)', tool_calendar('다음주_화'))
step('그 시간에 빈 회의실 찾기', '회의실 검색(화, 15시)', tool_find_room('다음주_화', '15시'))
step('방을 잡는다', '예약(B동 3호, 화 15시)', tool_book('B동 3호', '다음주_화', '15시'))
step('참석자들에게 알린다', '초대 발송(4명)', tool_invite(4))
step('정말 잡혔는지 마지막 확인', '예약 확인(R-1027)', tool_confirm('R-1027'))

print('\n최종 보고: 화요일 15시 B동 3호 예약 완료(R-1027), 4명 초대, 확정 상태 확인.')
assert len(trajectory) == 5, '이 일은 5단계 사슬이어야 한다'
print(f'✅ 궤적 길이 = {len(trajectory)}단계. 모든 문장이 도구로 확인한 사실')

## 3. 루프의 대가: 워밍업의 곱셈이 그대로 적용된다

루프의 스텝 하나하나(도구 선택·인자 작성·결과 해석)가 **실패할 수 있는 걸음**입니다.
이 5단계 예약 일도 걸음 정확도 90%라면:

In [ ]:
p = 0.9
n = len(trajectory)  # 5단계
success = p ** n
print(f'걸음 정확도 {p:.0%} × {n}단계 → 일의 성공률 = 0.9^{n} = {success:.4f} (약 {success*100:.0f}%)')
assert n == 5
assert abs(success - 0.5905) < 5e-4
print('✅ 유능해 보이는 루프도 다섯 걸음이면 59%. 되살리는 기술(4~6장)이 필요한 이유')

## 4. 오늘의 발견

- **챗봇 = 한 턴 함수** (외부 세계를 모름) / **에이전트 = LLM + 도구 + 루프** (확인된 사실 위에서 일함).
- 루프의 한 바퀴 = **관찰 → 생각 → 행동** (실무 용어로 ReAct 패턴), 스텝 기록 전체 = **궤적(trajectory)**.
- 행동도 결국 **다음 단어처럼 텍스트로 뽑는다** (4부 12장의 생성 루프가 원형). 그래서 확률이 붙는다.
- 루프의 스텝 = 워밍업의 '걸음'. 5단계 예약 일도 걸음 90%면 성공률 **0.9⁵ ≈ 59%**.

다음 장(2장 징검다리)에서 단계마다 정확도가 다른 진짜 사슬의 곱과 병목을 계산합니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**